In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# CONFIGURATION

num_iter = 10
split_folder = "astro_smote_splits"
save_name = "astrocyte_classification_results.csv"
importances_save_name = "astrocyte_feature_importances.csv"

# Feature Names
df_name_features = [
    'N_edg', 'Betw', 'Spec_ent', 'Alg_Conn', 'Assort', 'G_diam',
    'Br_dens', 'Br_l', 'Energy', 'Apl', 'Circ', 'FD',
    'Dir_std', 'q1', 'q2', 'q3', 'q4', 'Dir_ent', 'Order'
]

# Classes (Astrocyte specific)
class_names = ['AC', 'DM', 'LAT', 'VM', 'PC']
class_keys = ['ac', 'dm', 'lat', 'vm', 'pc']

# Storage
results = [] 
location_accuracies = {key: [] for key in class_keys + ['overall']}
all_importances = []

# ==========================================
# MAIN LOOP
# ==========================================
for i in range(num_iter):
    print(f"\n--- Processing Split {i} ---")
    
    # Load Split
    try:
        train_df = pd.read_csv(f"{split_folder}/train_split_{i}.csv")
        test_df = pd.read_csv(f"{split_folder}/test_split_{i}.csv")
    except FileNotFoundError:
        print("Error: Splits not found. Run 'generate_smote_splits.py' first.")
        break
        
    y_train = train_df['label_target'].values
    X_train = train_df.drop('label_target', axis=1).values
    y_test = test_df['label_target'].values
    X_test = test_df.drop('label_target', axis=1).values

    #  Train
    rf = RandomForestClassifier(n_estimators=100, max_depth=20, criterion='gini', random_state=0)
    clf = OneVsRestClassifier(rf)
    clf.fit(X_train, y_train)
    
    # Evaluate
    pred_test = clf.predict(X_test)
    score = accuracy_score(y_test, pred_test)
    print(f"Accuracy: {score:.4f}")
    
   
    print(f"Classification report for Split {i}")
    print(classification_report(y_test, pred_test, target_names=class_names, zero_division=0))
    
    print(f"Confusion Matrix for Split {i}:")
    print(confusion_matrix(y_test, pred_test))
   
    
    # Store Stats
    location_accuracies['overall'].append(score)
    for idx, key in enumerate(class_keys):
        mask = (y_test == idx)
        acc = accuracy_score(y_test[mask], pred_test[mask]) if np.sum(mask) > 0 else 0
        location_accuracies[key].append(acc)

    # Store Importances
    imps = [est.feature_importances_ for est in clf.estimators_]
    all_importances.append(np.mean(imps, axis=0) * 100)

# ==========================================
# FINAL RESULTS & PLOT
# ==========================================
mean_perf = np.mean(location_accuracies['overall'])
std_perf = np.std(location_accuracies['overall'])
results.append(["RandomForest", mean_perf, std_perf])

print("\n" + "="*40)
print("FINAL RESULTS (Mean ± Std over 10 splits)")
print("="*40)

# Calculate stats per location
summary_values = []
for key in class_keys + ['overall']:
    m = np.mean(location_accuracies[key])
    s = np.std(location_accuracies[key])
    summary_values.append(f"{m:.4f} ± {s:.4f}")


print(" | ".join([k.upper() for k in class_keys + ['overall']]))
print(" | ".join(summary_values))

# Save Results CSV
pd.DataFrame(results, columns=["Classifier", "Accuracy on Test Set", "Std"]).to_csv(save_name, index=False)
print(f"\nAccuracy results saved to: {save_name}")

# Importances
all_importances = np.array(all_importances)
final_mean = np.mean(all_importances, axis=0)
final_std = np.std(all_importances, axis=0)

idx = np.argsort(final_mean)[::-1]
sorted_names = np.array(df_name_features)[idx]
sorted_mean = final_mean[idx]
sorted_std = final_std[idx]

print("\nAggregated Feature Importances:")
for name, m, s in zip(sorted_names, sorted_mean, sorted_std):
    print(f"{name}: {m:.2f}% ± {s:.2f}")

# Save Importances CSV
pd.DataFrame({'Feature': sorted_names, 'Mean %': sorted_mean.round(2), 'Std': sorted_std.round(2)}).to_csv(importances_save_name, index=False)

# Plot
fig, ax = plt.subplots(figsize=(12, 8))
pd.Series(sorted_mean[::-1], index=sorted_names[::-1]).plot.barh(xerr=sorted_std[::-1], ax=ax, color='teal', edgecolor='black')
ax.set_title("Aggregated Feature Importances (Astrocyte)", fontsize=18)
ax.set_xlabel("Mean Decrease in Impurity (%)", fontsize=14)
sns.despine()
plt.tight_layout()
plt.show()